# Goal-scoring opportunity phase: defensive compactness and space control

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

For the goal-scoring opportunity phase, this notebook measures the defending team's compactness
(inter-player distances) and both teams' space control (Voronoi tessellation).

This notebook runs the shared pipeline (`pitchvision`: detection, tracking, pitch calibration,
team classification - see `00_pipeline_demo.ipynb` for a step-by-step walkthrough with sanity
checks) over **every clip in the `Goals` folder**, not just one - each clip gets its own saved
CSVs, and a per-clip failure (bad calibration, too few jersey samples) is skipped with a warning
rather than stopping the whole run.

**One thing the batch loop can't automate:** "defensive compactness" is defined from the
perspective of the team *conceding* the chance, and knowing which `team_id` that is for a given
clip needs a human to actually watch it - jersey colour alone doesn't say who's defending. So the
batch loop computes and saves BOTH teams' compactness/space-control for every clip (no need to
decide anything per clip to get that far); a separate section at the end lets you pick one clip,
identify its defending team by eye, and get the defending-team-specific numbers/plots the original
single-clip version of this notebook produced.

Batch results (one row per clip, both teams) are saved to `goals_all_clips_summary.csv`.

## 1. Setup

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/field-position-detection-dqda1g"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

# Put the package on sys.path directly, rather than relying on `pip install -e .`
# to register it - editable installs use a .pth/import-finder file that Python's
# site module only processes at interpreter startup, so one run mid-session (in
# an already-running Colab kernel) doesn't reliably become importable.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Drive and list clips

In [ ]:
from pitchvision import DriveConfig, list_videos, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")
goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/:")
for path in goal_videos:
    print(" ", os.path.basename(path))

## 3. Shared model setup (downloaded once)

Model weights don't need re-downloading per clip - only pitch calibration and team classification
are actually clip-specific. The detector/tracker themselves are still rebuilt fresh *inside* the
per-clip function below (see `process_goals_clip`) rather than reused across clips: Ultralytics'
`model.track(..., persist=True)` keeps tracker state (track IDs, Kalman filters) alive across calls
on the *same* model instance, which would leak track identity from one clip into the next if the
same `PlayerTracker` were reused - the same pattern `03_set_pieces.ipynb` already uses for exactly
this reason.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from pitchvision import (
    PITCH_LENGTH_M,
    PITCH_WIDTH_M,
    PitchKeypointDetector,
    PlayerBallDetector,
    PlayerTracker,
    SPORTS_DETECTION_CLASSES,
    TeamClassifier,
    TrackingPipeline,
    VideoFrames,
    collect_jersey_samples,
    compute_space_control,
    compute_team_compactness,
    download_pitch_keypoint_weights,
    download_player_detection_weights,
    draw_pitch,
    pitch_voronoi_cells,
    plot_jersey_color_samples,
    plot_voronoi,
    save_voronoi_frames,
)

pitch_weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)
keypoint_detector = PitchKeypointDetector(weights=pitch_weights_path, confidence=0.5)

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
print("Shared weights ready.")

## 4. Process every clip

Computes BOTH teams' compactness and space control for every clip - no "which team is defending"
decision needed to get this far. `silhouette_score` (see `04_validation.ipynb`) gives a
zero-labeling sanity check on each clip's team-colour split - low/negative means that clip's
jersey-colour clustering probably isn't finding real team structure, worth auditing with
`plot_jersey_color_samples` in the inspect-one-clip section below.

In [ ]:
def process_goals_clip(video_path, output_dir):
    frames = VideoFrames(video_path)
    first_frame = frames.read_frame(0)
    calibrator = keypoint_detector.calibrate(first_frame)

    detector = PlayerBallDetector(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    jersey_samples = collect_jersey_samples(video_path, detector, class_names=("player",), stride=15)
    jersey_colors = np.array([s.color for s in jersey_samples])
    team_classifier = TeamClassifier(n_clusters=2).fit(jersey_colors)

    cluster_labels = team_classifier.predict_from_colors(jersey_colors)
    silhouette = float("nan")
    if len(set(cluster_labels)) > 1:
        scaled = StandardScaler().fit_transform(jersey_colors)
        silhouette = silhouette_score(scaled, cluster_labels)

    tracker = PlayerTracker(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    pipeline = TrackingPipeline(
        tracker=tracker,
        calibrator=calibrator,
        team_classifier=team_classifier,
        team_eligible_class_names=("player",),
    )
    tracks_df = pipeline.run(video_path)  # a Goals clip is short enough to run in full

    compactness0 = compute_team_compactness(tracks_df, team_id=0)
    compactness1 = compute_team_compactness(tracks_df, team_id=1)
    space_df = compute_space_control(tracks_df)

    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    compactness0.to_csv(os.path.join(output_dir, f"{clip_name}_compactness_team0.csv"), index=False)
    compactness1.to_csv(os.path.join(output_dir, f"{clip_name}_compactness_team1.csv"), index=False)
    space_df.to_csv(os.path.join(output_dir, f"{clip_name}_space_control.csv"), index=False)

    pitch_area_m2 = PITCH_LENGTH_M * PITCH_WIDTH_M
    summary = {
        "clip": clip_name,
        "n_frames": tracks_df["frame"].nunique(),
        "jersey_cluster_silhouette": silhouette,
        "mean_stretch_team0_m": compactness0["stretch_index_m"].mean(),
        "mean_stretch_team1_m": compactness1["stretch_index_m"].mean(),
        "mean_space_pct_team0": 100 * space_df.get("team_0_area_m2", pd.Series(dtype=float)).mean() / pitch_area_m2,
        "mean_space_pct_team1": 100 * space_df.get("team_1_area_m2", pd.Series(dtype=float)).mean() / pitch_area_m2,
    }
    return {
        "summary": summary,
        "tracks_df": tracks_df,
        "compactness0": compactness0,
        "compactness1": compactness1,
        "space_df": space_df,
        "jersey_samples": jersey_samples,
        "team_classifier": team_classifier,
        "team_colors": {0: "yellow", 1: "red"},
    }

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)

results_by_clip = {}
summaries = []
failed = []
for video_path in goal_videos:
    clip_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f"Processing {clip_name}...")
    try:
        result = process_goals_clip(video_path, output_dir)
    except Exception as e:
        print(f"  SKIPPED ({type(e).__name__}: {e})")
        failed.append({"clip": clip_name, "error": str(e)})
        continue
    results_by_clip[clip_name] = result
    summaries.append(result["summary"])
    print(f"  done - {result['summary']['n_frames']} frames tracked")

summary_df = pd.DataFrame(summaries)
summary_path = os.path.join(output_dir, "goals_all_clips_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nProcessed {len(summaries)}/{len(goal_videos)} clips successfully. Summary saved to {summary_path}")
if failed:
    print("Skipped clips:", failed)
summary_df

## 5. Inspect one clip closely

Batch processing above computes both teams so it needs no per-clip decision, but the actual
goal-scoring-opportunity analysis is about the *defending* team specifically - that needs a human
to watch the clip and say which `team_id` conceded. Pick a clip from the batch results above; no
need to re-run the pipeline, this reuses the cached result.

In [ ]:
INSPECT_CLIP = next(iter(results_by_clip))  # change to inspect a different clip - see results_by_clip.keys()
print("Inspecting:", INSPECT_CLIP)

result = results_by_clip[INSPECT_CLIP]
tracks_df = result["tracks_df"]
compactness0, compactness1, space_df = result["compactness0"], result["compactness1"], result["space_df"]
jersey_samples, team_classifier = result["jersey_samples"], result["team_classifier"]
TEAM_COLORS = result["team_colors"]
print(f"silhouette score: {result['summary']['jersey_cluster_silhouette']:.3f}")

swatches = team_classifier.cluster_swatches
fig, axes = plt.subplots(1, len(swatches), figsize=(4 * len(swatches), 2))
for team_id, (ax, rgb) in enumerate(zip(axes, swatches)):
    ax.imshow([[rgb]])
    ax.set_title(f"team_id = {team_id}")
    ax.axis("off")
plt.show()

**Audit the fit before trusting it.** The scatter below shows every sampled point in (hue,
saturation) space, coloured by assigned cluster; the crop grid shows what those points actually
look like. Two visually separated blobs of roughly similar size = trustworthy. One tight blob plus
a handful of scattered outliers = the clustering isn't finding team identity - it's finding
whatever few samples looked different by chance.

In [ ]:
cluster_labels = team_classifier.predict_from_colors(np.array([s.color for s in jersey_samples]))
plot_jersey_color_samples(jersey_samples, cluster_labels, cluster_colors={0: "yellow", 1: "red"})

**Identify the defending team.** Look at the swatch colours above against the clip: the
goal-scoring opportunity phase is defined from the perspective of the team *conceding* the chance.
Set `DEFENDING_TEAM_ID` below to that team's `team_id`.

In [ ]:
DEFENDING_TEAM_ID = 0  # EDIT THIS based on the swatches above, for the clip named in INSPECT_CLIP

### Defensive compactness (inter-player distances)

`pitchvision.compactness.compute_team_compactness` gives one row per frame with the defending team's mean pairwise inter-player distance and "stretch index" (mean distance from the team centroid) - a lower value means a tighter, more compact defensive block.

In [ ]:
defending_compactness = compactness0 if DEFENDING_TEAM_ID == 0 else compactness1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(defending_compactness["frame"], defending_compactness["mean_pairwise_distance_m"], label="Mean pairwise distance (m)")
ax.plot(defending_compactness["frame"], defending_compactness["stretch_index_m"], label="Stretch index (m)")
ax.set_xlabel("Frame")
ax.set_ylabel("Metres")
ax.set_title(f"Defensive compactness over the phase - {INSPECT_CLIP} (team {DEFENDING_TEAM_ID})")
ax.legend()
plt.show()

print(defending_compactness[["mean_pairwise_distance_m", "stretch_index_m", "length_m", "width_m"]].describe())

### Space control (Voronoi diagrams)

`pitchvision.voronoi.pitch_voronoi_cells` tessellates the pitch by every tracked player's position (both teams): the region closer to a given player than to anyone else is credited to them. This is the standard simplified "space control" model in tactical analysis - it ignores player speed/orientation/reaction time, unlike more advanced pitch-control models, but is a well-established first-order approximation.

In [ ]:
# A representative frame: the one with the most on-pitch players tracked.
on_pitch = tracks_df[tracks_df["class_name"].isin(["player", "goalkeeper"]) & tracks_df["team_id"].notna()]
sample_frame = on_pitch.groupby("frame").size().idxmax()
frame_rows = on_pitch[on_pitch["frame"] == sample_frame]

positions = frame_rows[["pitch_x", "pitch_y"]].to_numpy()
team_ids = frame_rows["team_id"].to_numpy()
polygons = pitch_voronoi_cells(positions)

ax = draw_pitch()
plot_voronoi(ax, polygons, team_ids, team_colors=TEAM_COLORS)
for (x, y), team_id in zip(positions, team_ids):
    ax.scatter(x, y, color=TEAM_COLORS.get(team_id, "gray"), edgecolors="black", s=60, zorder=3)
plt.title(f"Space control at frame {sample_frame} - {INSPECT_CLIP}")
plt.show()

In [ ]:
pitch_area_m2 = PITCH_LENGTH_M * PITCH_WIDTH_M

fig, ax = plt.subplots(figsize=(10, 4))
for team_id in (0, 1):
    col = f"team_{team_id}_area_m2"
    if col in space_df.columns:
        ax.plot(space_df["frame"], 100 * space_df[col] / pitch_area_m2,
                 label=f"Team {team_id}", color=TEAM_COLORS.get(team_id, "gray"))
ax.set_xlabel("Frame")
ax.set_ylabel("% of pitch controlled")
ax.set_title(f"Space control over the phase - {INSPECT_CLIP}")
ax.legend()
plt.show()

defending_col = f"team_{DEFENDING_TEAM_ID}_area_m2"
if defending_col in space_df.columns:
    mean_pct = 100 * space_df[defending_col].mean() / pitch_area_m2
    print(f"Defending team (team_id={DEFENDING_TEAM_ID}) controlled {mean_pct:.1f}% of the pitch on average.")

### Optional: batch-export Voronoi diagrams for this clip

Off by default - `save_voronoi_frames` renders one PNG per frame, which adds up fast across many
clips, so this only ever runs for the single inspected clip above, not the whole batch.

**Build your own from a CSV.** This function (and everything else in `pitchvision.voronoi`/`pitchvision.compactness`) works on *any* DataFrame shaped like `tracks_df` - it doesn't need to come from a live pipeline run. A CSV needs at minimum these columns: `frame` (int), `class_name` (str), `team_id` (int, one per player - `None`/blank for anything not on a team), `pitch_x`, `pitch_y` (metres, top-left-origin pitch coordinates - see `pitchvision.calibration.PitchCalibrator` if you need to convert from pixel coordinates yourself). Load it with `pd.read_csv(...)` and pass it to `save_voronoi_frames`, `compute_space_control`, or `compute_team_compactness` exactly like `tracks_df` - no detection/tracking/calibration needed to re-generate diagrams from data you already have.

In [ ]:
EXPORT_VORONOI_FRAMES = False

if EXPORT_VORONOI_FRAMES:
    voronoi_dir = os.path.join(output_dir, f"{INSPECT_CLIP}_voronoi")
    saved_paths = save_voronoi_frames(tracks_df, voronoi_dir, stride=10, team_colors=TEAM_COLORS)
    print(f"Saved {len(saved_paths)} Voronoi diagrams to {voronoi_dir}")
else:
    print("Skipping PNG export (EXPORT_VORONOI_FRAMES is False).")

## Notes and limitations

- **Every clip in `Goals/` is processed**, not just one - a clip that fails (bad calibration, too
  few jersey samples for a 2-cluster fit) is skipped with its error printed to `failed`, rather
  than stopping the whole batch. Check `len(summaries)` against `len(goal_videos)` and the printed
  skip list before treating `goals_all_clips_summary.csv` as complete.
- The batch summary reports both teams; it does NOT pick a "defending" team automatically - that
  framing only exists in section 5, for whichever clip you're inspecting, since it needs a human
  to actually watch the clip.
- The Voronoi space-control model treats every player as controlling the region strictly closer to them by straight-line distance. It doesn't account for player speed, current motion, or reaction time the way more advanced "pitch control" models (e.g. Spearman et al.) do - two players equidistant from a point are credited equally even if one is already sprinting toward it and the other is facing the wrong way. Treat the area/percentage figures as a first-order approximation, not a physically-grounded probability of reaching the ball first.
- Goalkeepers are included in space control (they occupy real pitch space) but excluded from the defensive-compactness metric by default (`compute_team_compactness`'s `class_names` doesn't include `"goalkeeper"`), since including a goalkeeper anchored near their own goal line would distort an outfield defensive-line compactness measure.
- `jersey_cluster_silhouette` in the summary is a *zero-labeling* proxy for team-split quality (see `04_validation.ipynb`) - low/negative values are worth auditing with `plot_jersey_color_samples` in section 5, not necessarily discarding outright.